# 2 - Preparing the Occupancy Data

**Purpose:** This notebook serves to prepare the data for modelling, which includes seperating data into appropriate splits (training, validating and testing). The information for each forecast includes current headcount, headcounts 15 and 30 minutes earlier, time of day and day of the week.

**Splits:**
| Splits | Period |
|---|---|
| Training | March 2024–August 2025 |
| Validation | September–December 2025 |
| Final testing | January–March 2026 |




In [1]:
from pathlib import Path
import zipfile
import pandas as pd
from IPython.display import display

print("pandas version:", pd.__version__)

pandas version: 3.0.6


### Identify the Data and Chosen Building space
Decided to use the space (227f1068) as seen from the occupancy feasibility notebook, it offeres one of the highest usable percentage of data along with the highest variation in headcount.

In [2]:
#Locating DataSet
zip_path = Path("../data/DataSet2.zip")

csv_name = (
    "DataSet2_Env_and_Occupancy/"
    "5occupancySensor_MayToDec2024_9MRows/"
    "5occupancySensor_MayToDec2024_9MRows.csv"
)

#Defining Space ID
space_id = "227f1068-9e41-469f-bef0-275f27a0bdba"

print("Working folder:", Path.cwd())
print("ZIP found:", zip_path.exists())
print("Selected space:", space_id)

Working folder: c:\Users\chris\OneDrive\Desktop\MMA3001\Project\mma3001_project\notebooks
ZIP found: True
Selected space: 227f1068-9e41-469f-bef0-275f27a0bdba


### Recording/Defining the chosen splits
Splits were chosen to be chronological (considering the whole idea is to use the past to predict the future). Additionally, each period includes its start but excludes its end

In [3]:
train_start = pd.Timestamp("2024-03-01", tz="Australia/Melbourne")
validation_start = pd.Timestamp("2025-09-01", tz="Australia/Melbourne")
test_start = pd.Timestamp("2026-01-01", tz="Australia/Melbourne")
study_end = pd.Timestamp("2026-04-01", tz="Australia/Melbourne")

horizon = pd.Timedelta(minutes=30)
max_age = pd.Timedelta(minutes=5)
forecast_frequency = "15min"

print("Study starts:", train_start)
print("Validation starts:", validation_start)
print("Testing starts:", test_start)
print("Study ends (excluded):", study_end)

Study starts: 2024-03-01 00:00:00+11:00
Validation starts: 2025-09-01 00:00:00+10:00
Testing starts: 2026-01-01 00:00:00+11:00
Study ends (excluded): 2026-04-01 00:00:00+11:00


### Organising Data set to only include chosen space
Just filtering data to only include the rows containing the correct space

In [4]:
columns = ["floorspaceid", "collecteddate", "headcount", "occupancystatus"]
space_parts = []

with zipfile.ZipFile(zip_path) as zip_file:
    with zip_file.open(csv_name) as csv_file:
        for chunk in pd.read_csv(csv_file, usecols=columns, chunksize=200000):
            selected_rows = chunk[chunk["floorspaceid"] == space_id]
            space_parts.append(selected_rows)

raw_space = pd.concat(space_parts, ignore_index=True)

print("Raw records for the selected space:", len(raw_space))
print("Number of space IDs:", raw_space["floorspaceid"].nunique())

Raw records for the selected space: 1500394
Number of space IDs: 1


#### Converts Dates and heacounts, then checks them

In [5]:
raw_space["timestamp"] = pd.to_datetime(
    raw_space["collecteddate"],
    format="ISO8601",
    utc=True,
    errors="coerce"
)

raw_space["local_time"] = raw_space["timestamp"].dt.tz_convert(
    "Australia/Melbourne"
)

raw_space["headcount"] = pd.to_numeric(
    raw_space["headcount"], errors="coerce"
)

invalid_times = raw_space["timestamp"].isna().sum()
invalid_counts = raw_space["headcount"].isna().sum()
negative_counts = (raw_space["headcount"] < 0).sum()
non_integer_counts = (raw_space["headcount"].dropna() % 1 != 0).sum()

print("Invalid timestamps:", invalid_times)
print("Missing or non-numeric headcounts:", invalid_counts)
print("Negative headcounts:", negative_counts)
print("Non-integer or non-finite headcounts:", non_integer_counts)

assert len(raw_space) > 0, "No rows found. Check the space ID."
assert invalid_times == 0, "Stop: inspect invalid timestamps before continuing."
assert invalid_counts == 0, "Stop: inspect invalid headcounts before continuing."
assert negative_counts == 0, "Stop: inspect negative headcounts before continuing."
assert non_integer_counts == 0, "Stop: inspect non-integer headcounts before continuing."

Invalid timestamps: 0
Missing or non-numeric headcounts: 0
Negative headcounts: 0
Non-integer or non-finite headcounts: 0


### Keep the study period and Inspect repeated timestamps

In [6]:
space_data = raw_space[
    (raw_space["local_time"] >= train_start)
    & (raw_space["local_time"] < study_end)
].copy()

space_data = space_data.sort_values("timestamp")
space_data = space_data.reset_index(drop=True)

print("Records in the study period:", len(space_data))
print("First observation:", space_data["local_time"].min())
print("Last observation:", space_data["local_time"].max())
print("Extra rows at repeated timestamps:", space_data["timestamp"].duplicated().sum())

print("First five observations:")
display(space_data[["local_time", "headcount"]].head())

repeated_times = space_data[
    space_data["timestamp"].duplicated(keep=False)
]

print("All records involved in repeated timestamps:")
display(repeated_times[["local_time", "headcount", "occupancystatus"]])

Records in the study period: 1483464
First observation: 2024-03-03 15:06:30+11:00
Last observation: 2026-03-31 23:17:52+11:00
Extra rows at repeated timestamps: 2
First five observations:


,local_time,headcount
0,2024-03-03 15:06:30+11:00,0
1,2024-03-03 15:08:18+11:00,1
2,2024-03-03 15:08:50+11:00,0
3,2024-03-03 15:09:02+11:00,1
4,2024-03-03 15:09:42+11:00,0


All records involved in repeated timestamps:


,local_time,headcount,occupancystatus
56782,2024-04-05 09:13:42+11:00,0,RecentlyOccupied
56783,2024-04-05 09:13:42+11:00,0,NotOccupied
208917,2024-05-23 09:19:28+10:00,0,RecentlyOccupied
208918,2024-05-23 09:19:28+10:00,0,NotOccupied
